In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# cmfe(기존 LTAU 기반)와 cue4unc(MC Dropout / Deep Ensemble)의 fold별 성능을 비교해
# uncertainty 방법 간 성능 차이가 통계적으로 유의미한지 ANOVA로 검정한다.
# scaffold_results: DeepChem ScaffoldSplitter로 나눈 5-fold(8_Kd_scaffold_split.ipynb /
# 8_Ki_scaffold_split.ipynb, selectivity_df/scaffold_{Kd,Ki})로 학습한 결과를 비교한다
# (run_cue_scaffold_all_*.sh / run_cue_scaffold_mc_dropout.sh / run_cue_scaffold_ensemble.sh 참고).
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/scaffold_results'
cue4unc_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue4unc/scaffold_results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 세 가지 소스: (표시 이름, 결과 루트 경로)
model_sources = {
    'LTAU': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
    'MC Dropout': os.path.join(cue4unc_base, 'MC_Dropout', '{dataset}', '{target}', 'v1', 'CUE'),
    'Deep Ensemble': os.path.join(cue4unc_base, 'Deep_Ensemble', '{dataset}', '{target}', 'v1', 'CUE'),
}

# 그래프에 표시될 모델 순서
model_order = ['MC Dropout', 'Deep Ensemble', 'LTAU']

In [2]:
def load_fold_metrics(model_dir_template, dataset, target, task_type='regression'):
    """predictions_all.csv(5-fold 예측 전체)를 읽어 fold별 rmse/r2/pcc를 계산한다.

    주의: train.py::aggregate_metrics()의 fold 루프(정렬된 Fold 순서, GroundTruth/예측값에
    대한 동일한 isfinite 마스킹, 동일한 get_metrics 호출)를 한 글자도 다르지 않게 재현해야
    fold별 값의 평균이 기존 metrics.csv의 rmse/r2/pcc와 소수점까지 정확히 일치한다
    (실제로 8개 dataset/target x 3개 방법 x 3개 metric = 72개 항목 모두 일치함을 확인함).
    """
    path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'predictions_all.csv')
    if not os.path.exists(path):
        return None

    df_ys = pd.read_csv(path)
    folds = sorted(df_ys['Fold'].unique())
    model_name = 'mlp'

    rows = []
    for fold in folds:
        fd = df_ys[df_ys['Fold'] == fold]

        y_true = fd['GroundTruth'].values.astype(float)
        y_pred = fd[model_name].values.astype(float)

        mask = np.isfinite(y_true) & np.isfinite(y_pred)
        y_true_f = y_true[mask]
        y_pred_f = y_pred[mask]

        if y_true_f.size == 0:
            continue

        met = get_metrics(y_true_f, y_pred_f, task_type=task_type)
        rows.append({'Fold': fold, 'rmse': met['rmse'], 'r2': met['r2'], 'pcc': met['pcc']})

    return pd.DataFrame(rows) if rows else None

import sys
sys.path.append('/data/rlawlsgurjh/work/gitlab/CUE/cmfe')
from cue.utils.metric import get_metrics

raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            df_fold = load_fold_metrics(dir_template, d_name, target)
            if df_fold is None:
                continue
            for _, r in df_fold.iterrows():
                raw_rows.append({
                    'Dataset': d_name, 'Target': target, 'Model': model_name, 'Fold': r['Fold'],
                    'rmse': r['rmse'], 'r2': r['r2'], 'pcc': r['pcc'],
                })

df_folds = pd.DataFrame(raw_rows)
print(f"✅ Fold별 데이터 로드 완료: {len(df_folds)}개의 행이 수집되었습니다 (기대값: 2 x 4 x 3 x 5 = 120).")

✅ Fold별 데이터 로드 완료: 120개의 행이 수집되었습니다 (기대값: 2 x 4 x 3 x 5 = 120).


In [3]:
def plot_fold_boxplot(df_folds, dataset, target_name, metric_name, output_dir, model_order=model_order):
    """지정한 dataset/target/metric에 대해 모델별 fold 값(5개씩) 분포를 box plot으로 표시한다.
    (개별 fold 값은 stripplot으로 함께 겹쳐 표시해 표본 수를 눈으로 확인할 수 있게 한다.)
    X축에는 모델명을 표시하지 않는 대신, 색상-모델 매핑을 범례로 보여준다."""
    df_sub = df_folds[(df_folds['Dataset'] == dataset) & (df_folds['Target'] == target_name)]
    if df_sub.empty or metric_name not in df_sub.columns:
        return

    colors = ['#ef476f', '#ffd166', '#06d6a0']
    palette = {m: c for m, c in zip(model_order, colors)}

    fig, ax = plt.subplots(figsize=(6, 4))

    sns.boxplot(
        data=df_sub, x='Model', y=metric_name, order=model_order, hue='Model',
        palette=palette, ax=ax, width=0.5, showfliers=False, legend=False,
    )
    sns.stripplot(
        data=df_sub, x='Model', y=metric_name, order=model_order,
        color='black', alpha=0.7, size=5, ax=ax,
    )

    ax.set_title(f"{dataset} / {target_name} / {metric_name}")
    ax.set_xlabel("")
    ax.set_ylabel(metric_name.upper())
    ax.set_xticklabels([])
    ax.tick_params(axis='x', which='both', length=0)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    legend_handles = [plt.Rectangle((0, 0), 1, 1, facecolor=palette[m], edgecolor='black') for m in model_order]
    ax.legend(legend_handles, model_order, loc='best', fontsize=9, framealpha=0.9)

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{target_name}_{metric_name}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ 저장 완료: {save_path}")

In [4]:
plot_base_dir = os.path.join(cue_base, 'plots', 'uncertainty_anova')

for d_name in datasets:
    for target in targets:
        g_name = 'DIS' if d_name == 'DIS' else 'INH'
        output_path = os.path.join(plot_base_dir, g_name)
        for m in metrics_of_interest:
            plot_fold_boxplot(df_folds, d_name, target, m, output_path)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/uncertainty_anova/DIS/PI_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/uncertainty_anova/DIS/PI_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/uncertainty_anova/DIS/PI_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/uncertainty_anova/DIS/Ssel_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/uncertainty_anova/DIS/Ssel_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/uncertainty_anova/DIS/Ssel_pcc.png


... (14 outputs left out)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/uncertainty_anova/INH/WS(alpha=2)_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/uncertainty_anova/INH/RS(k=2)_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/uncertainty_anova/INH/RS(k=2)_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/scaffold_results/plots/uncertainty_anova/INH/RS(k=2)_pcc.png
